# Why is `improve_decomposition_study.py` slow? — a profile

Measures where the time actually goes, then derives a config for **8 CPUs + 1 T4**.

The headline suspect is **not** `img_batch`. LBFGS's cost is
`n_iter x lbfgs_max_iter x (line-search factor)` **forward+backward passes over every
train image**. With the study's `n_iter=500, lbfgs_max_iter=40` that is on the order of
**25,000 full-batch passes** per run — before any consideration of batching. Each of the
experiments below isolates one factor:

| # | question | knob |
|---|---|---|
| 1 | How many forward+backward passes does one outer step really cost? | `lbfgs_max_iter` |
| 2 | When does the loss actually plateau — is the budget wasted? | `n_iter` |
| 3 | Does a bigger `img_batch` help, and what does it cost in memory? | `img_batch` |
| 4 | How does cost scale with resolution? | `downsample` |
| 5 | Do parallel workers help on ONE GPU? | `--workers` |

> Run this **on the VM (T4, 16 GB)** for numbers that transfer. The defaults below are
> deliberately small so it also runs on a 4 GB laptop GPU; raise `N_IMAGES` / lower
> `DOWNSAMPLE` on the T4 to match the real study.


In [ ]:
import os
import sys, os, time, json, subprocess, tempfile
from pathlib import Path
import numpy as np, torch, pandas as pd
import matplotlib.pyplot as plt

# Repo root, found by walking up from the CWD. Notebooks live in notebooks/, but the
# code and the relative data paths below are relative to the repo root, so chdir there.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "idr").is_dir())
os.chdir(REPO)
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
os.environ.setdefault('WANDB_MODE', 'disabled')

from idr.data.geometry import make_proxy_geometry
from idr.data.scene_io import load_scene
from idr.config import DEFAULT_CFG
from idr.optim.models.ct_sh import _optimize_ct_sh

SCENE      = Path('results/3dfront-batch/datasets/1f19c3ef_v2/ct-ct_sh-frOn_env')
DOWNSAMPLE = 4        # profile resolution. The real study uses 2 (256^2) -> set 2 on the T4.
N_IMAGES   = 32       # train images. Real study uses 100 -> set 100 on the T4.
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# the real study's settings, for reference / extrapolation
STUDY_N_ITER, STUDY_MAX_ITER, STUDY_IMG_BATCH = 500, 40, 8
STUDY_N_TRAIN, STUDY_DS = 100, 2

if DEVICE == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  {p.total_memory/1e9:.1f} GB | CPUs: {os.cpu_count()}')

sc = load_scene(SCENE, gt_npy=True)
BASE_RES = sc['H']
def _strd(a, ds): return np.ascontiguousarray(a[::ds, ::ds])

def make_inputs(ds, k):
    N, mask = _strd(sc['normals_np'], ds), _strd(sc['mask_np'], ds)
    alb, met, rou = _strd(sc['albedo_np'], ds), _strd(sc['metallic_np'], ds), _strd(sc['roughness_np'], ds)
    imgs = [_strd(im, ds) for im in sc['images'][:k]]
    shs  = [np.asarray(s, np.float32) for s in sc['sh_coeffs'][:k]]
    Nhw, frag, mhw, cam = make_proxy_geometry(N, mask, 60, 2, DEVICE, torch.float32)
    return dict(imgs=imgs, Nhw=Nhw, frag=frag, mhw=mhw, cam=cam, met=met, rou=rou,
                alb=alb, shs=shs, M=int(mask.sum()), res=N.shape[0])

INP = make_inputs(DOWNSAMPLE, N_IMAGES)
print(f'profiling at {INP["res"]}^2 ({INP["M"]} masked px), {len(INP["imgs"])} images')


## The measurement harness

`timed_run` runs the **real** `_optimize_ct_sh` and reports wall time, peak GPU memory, and
the number of **LBFGS closure evaluations** — one closure = one forward+backward over the
selected images. Counting them is the only honest way to see the `n_iter x max_iter` blow-up,
because the line search adds evaluations on top of `max_iter`.

In [ ]:
_orig_step = torch.optim.LBFGS.step
_CNT = {'n': 0}

def _counting_step(self, closure):
    def wrapped():
        _CNT['n'] += 1
        return closure()
    return _orig_step(self, wrapped)

torch.optim.LBFGS.step = _counting_step          # count closure evals globally

def timed_run(inp, n_iter, max_iter, img_batch=0, log_every=10_000, **extra):
    cfg = {**DEFAULT_CFG, 'optimizer': 'LBFGS', 'n_iter': n_iter, 'lbfgs_max_iter': max_iter,
           'log_every': log_every, 'loss': 'L2', 'sh_order': 2, 'double': False,
           'tr_albedo': 'sigmoid', 'tr_metallic': 'sigmoid', 'tr_roughness': 'sigmoid',
           'init_roughness_zero': True, 'lambda_tv': 0.0, 'lambda_metallic_binarize': 0.0,
           'img_batch': img_batch, **extra}
    _CNT['n'] = 0
    if DEVICE == 'cuda':
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = _optimize_ct_sh(inp['imgs'], inp['Nhw'], inp['frag'], inp['mhw'], inp['cam'],
                          inp['met'], inp['rou'], cfg, gt_sh_coeffs=inp['shs'], gt_albedo=inp['alb'])
    if DEVICE == 'cuda': torch.cuda.synchronize()
    el = time.perf_counter() - t0
    peak = torch.cuda.max_memory_allocated()/1e9 if DEVICE == 'cuda' else 0.0
    return dict(wall_s=el, closures=_CNT['n'], peak_gb=peak, history=out[5])

timed_run(INP, 2, 5)        # warm up CUDA + the GGX LUT
print('harness ready')


## 0. Task breakdown — where does the wall time actually go?

Times the **real** code path with `cuda.synchronize()` around each phase, by swapping in an
instrumented `_opt_step`. This is the "which task takes how long" answer. (A naive
`torch.profiler` run is misleading here: `Optimizer.step#LBFGS.step` is a *scope* that
swallows the nested forward/backward, so it reports impossible things like >100% GPU busy.)

Phases:
* **forward** / **backward** inside the LBFGS closure — the actual work
* **LBFGS internal** — two-loop recursion + line-search bookkeeping over the flat param
  vector. Driven by `history_size`; it is a long chain of tiny launch-bound kernels.
* **extra reporting forward** — `_opt_step` runs one more `no_grad` forward per outer step
  purely to report the loss.
* **setup + final shadings + misc**

In [ ]:
import raw_optimizer.synthetic_ct_dataset as _S
_T = {}
def _reset(): _T.update(fwd=0.0, bwd=0.0, step=0.0, extra=0.0, nclo=0, nout=0)
_ORIG_STEP = _S._opt_step

def _timed_opt_step(opt, forward_fn, cfg):
    sy = torch.cuda.synchronize if DEVICE == 'cuda' else (lambda: None)
    def closure():
        opt.zero_grad()
        sy(); a = time.perf_counter()
        loss, *_ = forward_fn()
        sy(); b = time.perf_counter()
        loss.backward()
        sy(); c = time.perf_counter()
        _T['fwd'] += b-a; _T['bwd'] += c-b; _T['nclo'] += 1
        return loss
    sy(); s0 = time.perf_counter()
    try: opt.step(closure)
    except (IndexError, TypeError): opt.state.clear()
    sy(); _T['step'] += time.perf_counter()-s0; _T['nout'] += 1
    with torch.no_grad():                       # _opt_step's extra reporting forward
        sy(); f0 = time.perf_counter()
        r = forward_fn()
        sy(); _T['extra'] += time.perf_counter()-f0
    return r

_S._opt_step = _timed_opt_step
BD_OUTER, BD_MI = 10, 20
_reset(); _r = timed_run(INP, BD_OUTER, BD_MI)          # img_batch=0 (full batch)
wall = _r['wall_s']
internal = _T['step'] - _T['fwd'] - _T['bwd']
misc = wall - _T['step'] - _T['extra']
rows = [('forward  (in closure)', _T['fwd'], _T['nclo']),
        ('backward (in closure)', _T['bwd'], _T['nclo']),
        ('LBFGS internal (history/line-search)', internal, _T['nout']),
        ('extra reporting forward (no_grad)', _T['extra'], _T['nout']),
        ('setup + final shadings + misc', misc, 1)]
print(f"=== TASK BREAKDOWN ({len(INP['imgs'])} imgs @ {INP['res']}^2, "
      f"{BD_OUTER} outer x max_iter={BD_MI}, img_batch=full) ===")
print(f"{'task':38} {'sec':>7} {'% wall':>7} {'calls':>7} {'ms/call':>9}")
for n, v, c in rows:
    print(f'{n:38} {v:7.2f} {100*v/wall:7.1f} {c:7d} {v/max(c,1)*1000:9.1f}')
print(f"{'TOTAL wall':38} {wall:7.2f} {100.0:7.1f}")
print(f"\nclosures = {_T['nclo']} ({_T['nclo']/_T['nout']:.1f} per outer step)   "
      f"backward/forward = {_T['bwd']/max(_T['fwd'],1e-9):.2f}x")
print(f"=> forward+backward is {100*(_T['fwd']+_T['bwd'])/wall:.0f}% of wall; "
      f"the closure COUNT is therefore the main lever (see cells 1-2).")

fig, ax = plt.subplots(figsize=(7, 3.2))
labels = [r[0] for r in rows][::-1]; vals = [r[1] for r in rows][::-1]
ax.barh(labels, vals, color=['#999', '#DD8452', '#8172B3', '#C44E52', '#4C72B0'])
ax.set_xlabel('seconds'); ax.set_title('where the wall time goes', fontsize=10)
ax.grid(axis='x', alpha=0.3); plt.tight_layout(); plt.show()
_S._opt_step = _ORIG_STEP


## 1. The `n_iter x lbfgs_max_iter` multiplier — the real cost driver

One `opt.step()` runs up to `lbfgs_max_iter` LBFGS iterations, **each** needing at least one
closure (forward+backward over all train images), plus extra evaluations from the
`strong_wolfe` line search. So the total work is
`n_iter x max_iter x line_search_factor` full passes.

In [ ]:
rows = []
for mi in (5, 10, 20, 40):
    r = timed_run(INP, 5, mi)
    rows.append(dict(max_iter=mi, closures=r['closures'], per_outer=r['closures']/5,
                     ls_factor=r['closures']/5/mi, wall_s=r['wall_s'],
                     ms_per_closure=r['wall_s']/max(r['closures'],1)*1000))
t1 = pd.DataFrame(rows)
print(t1.round(2).to_string(index=False))

ls = t1['ls_factor'].mean()
per_closure_ms = t1['ms_per_closure'].median()
study_closures = STUDY_N_ITER * STUDY_MAX_ITER * ls
print(f'\nline-search factor ~{ls:.2f}x on top of max_iter')
print(f'=> the study config (n_iter={STUDY_N_ITER}, max_iter={STUDY_MAX_ITER}) costs '
      f'~{study_closures:,.0f} forward+backward passes PER RUN')
print(f'   at this profile size that alone would be '
      f'~{study_closures*per_closure_ms/1000/60:.1f} min/run '
      f'(and the real study is bigger: {STUDY_N_TRAIN} imgs @ {BASE_RES//STUDY_DS}^2)')


## 2. Where does the loss actually plateau?

If the loss flattens after a few hundred closures, then most of those ~25,000 passes are
buying nothing. This is the single biggest lever — far bigger than `img_batch`.

In [ ]:
# Long enough to actually see a plateau. If the curve is still descending at the end,
# the "wasted budget" question is UNANSWERED - this cell says so instead of quoting a
# bogus ratio against a not-yet-converged final loss.
PLATEAU_OUTER, PLATEAU_MI = 200, 20
r = timed_run(INP, PLATEAU_OUTER, PLATEAU_MI, log_every=1)
h = np.array(r['history'], dtype=float)
cpo = r['closures'] / PLATEAU_OUTER
x = np.arange(len(h)) * cpo
final = h[-1]
print(f'spent {r["closures"]:,} closures over {PLATEAU_OUTER} outer steps -> loss {final:.4e}')

print('\ndiminishing returns - loss at fractions of THIS budget:')
for f in (0.05, 0.10, 0.25, 0.50, 1.00):
    i = min(int(f * (len(h) - 1)), len(h) - 1)
    print(f'  {f:5.0%} ({x[i]:7,.0f} closures): {h[i]:.4e}   ({h[i]/final:5.2f}x final)')

def reach(frac):
    tgt = final * (1 + frac)
    idx = int(np.argmax(h <= tgt)) if (h <= tgt).any() else len(h) - 1
    return x[idx], idx
c10, i10 = reach(0.10); c01, i01 = reach(0.01)
print(f'\nwithin 10% of final: ~{c10:,.0f} closures (step {i10}/{len(h)-1})')
print(f'within  1% of final: ~{c01:,.0f} closures (step {i01}/{len(h)-1})')

PLATEAUED = i01 < 0.9 * (len(h) - 1)
if not PLATEAUED:
    print('\nWARNING: the 1% threshold is only reached at the very END of this run, i.e. the')
    print('  loss is STILL DESCENDING and this run never plateaued. Do NOT conclude the study')
    print('  budget is wasted from this - raise PLATEAU_OUTER and re-run.')
else:
    print(f'\n=> plateau reached well before the end. Study budget '
          f'(~{STUDY_N_ITER*STUDY_MAX_ITER*ls:,.0f}) is '
          f'~{STUDY_N_ITER*STUDY_MAX_ITER*ls/max(c01,1):.0f}x more than needed for within-1%.')
    print(f'   suggested n_iter at max_iter={STUDY_MAX_ITER}: '
          f'~{int(np.ceil(c01/(STUDY_MAX_ITER*ls)))} (vs {STUDY_N_ITER})')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].semilogy(x, h, lw=1.8); ax[0].set_xlabel('cumulative closure evals (fwd+bwd)')
ax[0].set_ylabel('data loss'); ax[0].set_title('convergence vs actual work', fontsize=10)
for c, lb, col in ((c10, 'within 10%', '#DD8452'), (c01, 'within 1%', '#55A868')):
    ax[0].axvline(c, ls='--', lw=1, color=col, label=f'{lb} @ {c:,.0f}')
ax[0].axvline(STUDY_N_ITER*STUDY_MAX_ITER*ls, ls=':', lw=1.5, color='#C44E52',
              label=f'study budget ~{STUDY_N_ITER*STUDY_MAX_ITER*ls:,.0f}')
ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3, which='both')
g = np.gradient(np.log10(np.maximum(h, 1e-30)), np.maximum(x, 1e-9))
ax[1].plot(x[1:], -g[1:] * 1000, lw=1.5)
ax[1].set_xlabel('cumulative closures'); ax[1].set_ylabel('loss decades per 1k closures')
ax[1].set_title('marginal return (flat ~0 => stop)', fontsize=10); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 3. `img_batch` — the hypothesis under test

`img_batch>0` splits each closure into `ceil(K/img_batch)` chunked forward+backward passes
(gradient accumulation; the summed gradient equals the full-batch one, so it is safe under
LBFGS). More chunks = more kernel launches and Python overhead, but a smaller autograd graph.
`img_batch=0` = one full-batch pass = fewest launches, most memory.

**Expectation:** wall time should *fall* and peak memory *rise* as `img_batch` grows — i.e.
use the largest value that fits. The point of the table is to find where it stops helping.

In [ ]:
K = len(INP['imgs'])
cands = [b for b in (2, 4, 8, 16, 32, 64, 128) if b < K] + [0]     # 0 = full batch
rows = []
for b in cands:
    try:
        r = timed_run(INP, 3, 10, img_batch=b)
        chunks = 1 if b == 0 else int(np.ceil(K / b))
        rows.append(dict(img_batch=('full' if b == 0 else b), chunks_per_closure=chunks,
                         wall_s=r['wall_s'], s_per_outer=r['wall_s']/3, peak_gb=r['peak_gb']))
    except torch.cuda.OutOfMemoryError:
        rows.append(dict(img_batch=('full' if b == 0 else b), chunks_per_closure=np.nan,
                         wall_s=np.nan, s_per_outer=np.nan, peak_gb=np.nan))
        torch.cuda.empty_cache()
t3 = pd.DataFrame(rows)
print(t3.round(3).to_string(index=False))

ok = t3.dropna()
if len(ok):
    best = ok.loc[ok['s_per_outer'].idxmin()]
    print(f"\nfastest: img_batch={best['img_batch']}  "
          f"({best['s_per_outer']:.2f}s/outer, {best['peak_gb']:.2f} GB peak)")
    slow = ok['s_per_outer'].max()
    print(f"speedup vs the slowest setting: {slow/best['s_per_outer']:.2f}x")

fig, ax1 = plt.subplots(figsize=(7, 4))
lbl = [str(v) for v in ok['img_batch']]
ax1.bar(range(len(ok)), ok['s_per_outer'], color='#4C72B0', label='s / outer step')
ax1.set_xticks(range(len(ok))); ax1.set_xticklabels(lbl); ax1.set_xlabel('img_batch')
ax1.set_ylabel('s / outer step', color='#4C72B0')
ax2 = ax1.twinx(); ax2.plot(range(len(ok)), ok['peak_gb'], 'o-', color='#C44E52')
ax2.set_ylabel('peak GPU GB', color='#C44E52')
plt.title('img_batch: time vs memory', fontsize=10); plt.tight_layout(); plt.show()


## 3b. LBFGS `history_size` — the other launch-bound tax

The two-loop recursion touches the whole flat parameter vector `history_size` times per
iteration, as a long chain of tiny kernels. torch defaults to **100**. Measured on this repo
it was ~47% of wall at 100 vs ~26% at 20 — **with an identical final loss**.

`_make_optimizer` in `raw_optimizer/synthetic_ct_dataset.py` is where this is set. Also check
`tolerance_grad` / `tolerance_change` there: setting them to **0 disables LBFGS's early stop**,
forcing the full `max_iter` inner iterations every outer step even after convergence.

In [ ]:
_ORIG_MK = _S._make_optimizer
_HIST = [None]
def _mk(params, cfg):
    if _optimizer_name_safe(cfg) == 'LBFGS' and _HIST[0] is not None:
        import inspect
        src = inspect.getsource(_ORIG_MK)
        kw = dict(lr=cfg['lr'], max_iter=cfg['lbfgs_max_iter'], line_search_fn='strong_wolfe',
                  history_size=_HIST[0])
        if 'tolerance_grad' in src:            # mirror the repo's early-stop setting
            kw.update(tolerance_grad=0, tolerance_change=0)
        return torch.optim.LBFGS(params, **kw)
    return _ORIG_MK(params, cfg)

def _optimizer_name_safe(cfg):
    return str(cfg.get('optimizer', 'LBFGS')).upper()

_S._make_optimizer = _mk
rows = []
for h in (100, 50, 20, 10):
    _HIST[0] = h
    _reset(); _S._opt_step = _timed_opt_step
    r = timed_run(INP, 6, 20)
    internal = _T['step'] - _T['fwd'] - _T['bwd']
    rows.append(dict(history_size=h, wall_s=r['wall_s'], lbfgs_internal_s=internal,
                     pct_wall=100*internal/r['wall_s'], final_loss=r['history'][-1]))
    _S._opt_step = _ORIG_STEP
_HIST[0] = None; _S._make_optimizer = _ORIG_MK
t3b = pd.DataFrame(rows)
print(t3b.round(4).to_string(index=False))
base = t3b[t3b.history_size == 100]['wall_s'].iloc[0]
t3b['speedup'] = base / t3b['wall_s']
print('\nspeedup vs history_size=100:')
print(t3b[['history_size', 'speedup', 'final_loss']].round(4).to_string(index=False))
print('\nPick the smallest history whose final_loss is unchanged.')


## 4. Resolution scaling

The study runs at `--study_downsample 2` (256^2) and the resolution sweep goes to full 512^2.
This is how the per-pass cost and memory scale.

In [ ]:
rows = []
for ds in (16, 8, 4, 2, 1):
    if BASE_RES // ds > 512: continue
    try:
        inp = make_inputs(ds, N_IMAGES)
        r = timed_run(inp, 2, 5)
        rows.append(dict(downsample=ds, res=inp['res'], masked_px=inp['M'],
                         ms_per_closure=r['wall_s']/max(r['closures'],1)*1000,
                         peak_gb=r['peak_gb']))
        del inp; torch.cuda.empty_cache()
    except torch.cuda.OutOfMemoryError:
        rows.append(dict(downsample=ds, res=BASE_RES//ds, masked_px=np.nan,
                         ms_per_closure=np.nan, peak_gb=np.nan)); torch.cuda.empty_cache()
t4 = pd.DataFrame(rows)
print(t4.round(3).to_string(index=False))
print(f'\n(profile used {N_IMAGES} images; the real study uses {STUDY_N_TRAIN} '
      f'-> multiply ms/closure by ~{STUDY_N_TRAIN/N_IMAGES:.1f})')


## 5. Do parallel workers help on ONE GPU?

The study parallelizes runs across processes. At 31^2 that was a big win (GPU ~12% idle,
CPU-bound). At 256^2 each worker does real GPU work, so they contend for **one** T4 — more
workers may not raise throughput, and each costs GPU memory.

This launches N *independent* processes doing the same fixed decomposition and measures
total wall time, giving true throughput (runs/min) vs worker count.

In [ ]:
WORKER_SCRIPT = tempfile.mktemp(suffix='_pw.py')
Path(WORKER_SCRIPT).write_text(f"""
import sys, os, time, numpy as np, torch
os.environ['WANDB_MODE']='disabled'
sys.path.insert(0, r'{REPO}')
from pathlib import Path
from idr.data.geometry import make_proxy_geometry
from idr.data.scene_io import load_scene
from idr.config import DEFAULT_CFG
from idr.optim.models.ct_sh import _optimize_ct_sh
ds, K = {DOWNSAMPLE}, {N_IMAGES}
sc = load_scene(Path(r'{SCENE}'), gt_npy=True)
s = lambda a: np.ascontiguousarray(a[::ds, ::ds])
Nhw, frag, mhw, cam = make_proxy_geometry(s(sc['normals_np']), s(sc['mask_np']), 60, 2, 'cuda', torch.float32)
cfg = {{**DEFAULT_CFG, 'optimizer':'LBFGS','n_iter':6,'lbfgs_max_iter':10,'log_every':10000,
        'loss':'L2','sh_order':2,'double':False,'tr_albedo':'sigmoid','tr_metallic':'sigmoid',
        'tr_roughness':'sigmoid','init_roughness_zero':True,'lambda_tv':0,
        'lambda_metallic_binarize':0,'img_batch':0}}
t0=time.perf_counter()
_optimize_ct_sh([s(i) for i in sc['images'][:K]], Nhw, frag, mhw, cam,
                s(sc['metallic_np']), s(sc['roughness_np']), cfg,
                gt_sh_coeffs=[np.asarray(x,np.float32) for x in sc['sh_coeffs'][:K]],
                gt_albedo=s(sc['albedo_np']))
torch.cuda.synchronize()
print(time.perf_counter()-t0)
""")

def throughput(n):
    t0 = time.perf_counter()
    ps = [subprocess.Popen([sys.executable, WORKER_SCRIPT], stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL, cwd=str(REPO)) for _ in range(n)]
    for p in ps: p.wait()
    el = time.perf_counter() - t0
    return el, n/el*60

WORKER_COUNTS = [1, 2, 4, 8]
rows = []
for n in WORKER_COUNTS:
    el, tp = throughput(n)
    rows.append(dict(workers=n, wall_s=el, runs_per_min=tp))
    print(f'workers={n}: {el:6.1f}s total -> {tp:5.2f} runs/min', flush=True)
t5 = pd.DataFrame(rows)
t5['speedup_vs_1'] = t5['runs_per_min'] / t5['runs_per_min'].iloc[0]
print()
print(t5.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(t5['workers'], t5['speedup_vs_1'], 'o-', lw=2, label='measured')
ax.plot(t5['workers'], t5['workers'], '--', color='0.6', label='ideal (linear)')
ax.set_xlabel('parallel workers (1 GPU)'); ax.set_ylabel('throughput speedup')
ax.set_title('does parallelism help on one GPU?', fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
os.unlink(WORKER_SCRIPT)


## 6. Verdict + recommended config

Read the numbers above, not this text — but the structure of the answer is fixed:

* **`n_iter x lbfgs_max_iter` dominates.** It sets the number of full forward+backward passes
  (x ~1.25 for the line search). Cell 2 shows where the loss plateaus; everything past that
  is pure waste, and it is usually an order of magnitude. Cut `n_iter` first.
* **`img_batch` is a second-order effect** (cell 3): it changes launch overhead and memory,
  not the number of passes. Bigger *is* better until it OOMs — so the right value is "the
  largest that fits", which on a 16 GB T4 at 256^2 is likely full batch (`0`).
* **Workers ≠ free** (cell 5): with one T4 the GPU is shared. If speedup flattens at 2-3,
  extra workers only add memory pressure. 8 CPUs do not mean 8 useful workers here.


In [ ]:
print('=== measured summary ===')
try:
    print(f'line-search factor            : {ls:.2f}x  (closures = n_iter * max_iter * this)')
    print(f'study budget (500 x 40)       : ~{STUDY_N_ITER*STUDY_MAX_ITER*ls:,.0f} fwd+bwd passes/run')
    c10, _ = reach(0.10); c01, _ = reach(0.01)
    print(f'closures to within 10% / 1%   : ~{c10:,.0f} / ~{c01:,.0f}')
    if PLATEAUED:
        print(f'=> budget is ~{STUDY_N_ITER*STUDY_MAX_ITER*ls/max(c01,1):.0f}x more than needed '
              f'for within-1% of final loss')
        print(f'=> suggested n_iter at max_iter={STUDY_MAX_ITER}: '
              f'~{int(np.ceil(c01/(STUDY_MAX_ITER*ls)))} (vs {STUDY_N_ITER})')
    else:
        print('=> probe did NOT converge: cannot claim the budget is wasted.')
        print('   Raise PLATEAU_OUTER in cell 2 and re-run before cutting n_iter.')
except NameError:
    print('(run cells 1-2 first)')
try:
    ok = t3.dropna(); best = ok.loc[ok['s_per_outer'].idxmin()]
    print(f'best img_batch                : {best["img_batch"]} '
          f'({best["peak_gb"]:.2f} GB peak at {INP["res"]}^2 / {N_IMAGES} imgs)')
except Exception:
    pass
try:
    b = t5.loc[t5['runs_per_min'].idxmax()]
    print(f'best worker count             : {int(b["workers"])} '
          f'({b["speedup_vs_1"]:.2f}x vs 1 worker)')
except Exception:
    pass
print('\nNB: extrapolate memory/time to the real study size '
      f'({STUDY_N_TRAIN} imgs @ {BASE_RES//STUDY_DS}^2) before committing.')
